# Step 6 — Build Docker Image, Serve, and Test



## 6.1 — Copy model into the build context

In [ ]:
import subprocess, os

MODEL_SRC  = "/home/prashant.takale/icml/4_models/qwen3.5-4b-final-combo"
BUILD_DIR  = "/home/prashant.takale/icml/34_final_combo"
MODEL_DST  = os.path.join(BUILD_DIR, "model")

print(f"Copying {MODEL_SRC} → {MODEL_DST} ...", flush=True)
result = subprocess.run(
    ["cp", "-r", MODEL_SRC, MODEL_DST],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("ERROR:", result.stderr)
else:
    print("Done.", flush=True)

# Check what's in the model dir
print("\nModel directory contents:")
for f in sorted(os.listdir(MODEL_DST)):
    sz = os.path.getsize(os.path.join(MODEL_DST, f))
    print(f"  {f:45s}  {sz/1e6:.1f} MB")

## 6.2 — Build Docker image

In [ ]:
# Build the image. Uses Dockerfile in BUILD_DIR.
# Note: vLLM base image version is PINNED at 0.19.0 — do NOT upgrade.

print("Building Docker image final-combo:latest ...", flush=True)
print("  This takes ~3-5 min (model copy into image layer).", flush=True)

result = subprocess.run(
    ["sudo", "docker", "build", "-t", "final-combo:latest", "."],
    cwd=BUILD_DIR,
    capture_output=False,   # stream output to notebook
)
print(f"\nDocker build exit code: {result.returncode}")
if result.returncode == 0:
    print(" Image built successfully")
else:
    print(" Build failed — check output above")

## 6.3 — Run container

In [ ]:
# Remove any old test container first

subprocess.run(["sudo", "docker", "rm", "-f", "combotest"],
               capture_output=True)

print("Starting container ...", flush=True)
result = subprocess.run(
    [
        "sudo", "docker", "run", "-d",
        "--name", "combotest",
        "--runtime=nvidia",
        "-e", "NVIDIA_VISIBLE_DEVICES=all",
        "-p", "8080:8080",
        "final-combo:latest",
    ],
    capture_output=True, text=True
)
print("Container ID:", result.stdout.strip())
if result.returncode != 0:
    print("ERROR:", result.stderr)

In [ ]:
# Wait for vLLM to become ready (polls /ping every 5s, timeout 15 min)

import time, urllib.request, urllib.error

print("Waiting for vLLM to be ready (this takes ~3-5 min) ...", flush=True)
deadline = time.time() + 900
while time.time() < deadline:
    try:
        with urllib.request.urlopen("http://localhost:8080/ping", timeout=3) as r:
            if r.status == 200:
                print("\n /ping → 200  (server is ready)", flush=True)
                break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(5)
else:
    print("\n Timeout waiting for server — check docker logs combotest")

## 6.4 — Smoke tests

In [ ]:
import json, urllib.request

BASE = "http://localhost:8080"

def post(path, payload):
    data = json.dumps(payload).encode()
    req  = urllib.request.Request(
        BASE + path, data=data,
        headers={"Content-Type": "application/json"}
    )
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.loads(r.read())

In [ ]:
# Test 1: raw completion via /invocations

print("=== /invocations (prompt path) ===")
resp = post("/invocations", {
    "prompt": "The capital of France is",
    "max_tokens": 10,
    "temperature": 0,
})
print(json.dumps(resp, indent=2))

In [ ]:
# Test 2: chat completion via /invocations

print("=== /invocations (messages path) ===")
resp = post("/invocations", {
    "messages": [{"role": "user", "content": "What is 2 + 2?"}],
    "max_tokens": 20,
    "temperature": 0,
})
print(json.dumps(resp, indent=2))

In [ ]:
# Test 3: check MTP speculative acceptance length from logs

print("=== MTP acceptance check ===")
result = subprocess.run(
    ["sudo", "docker", "logs", "--tail", "40", "combotest"],
    capture_output=True, text=True
)
# look for acceptance length lines (vLLM prints these periodically)
for line in result.stdout.splitlines():
    if any(kw in line for kw in ["acceptance", "spec", "draft", "mtp", "MTP", "accepted"]):
        print(line)
print("\n(MTP acceptance length > 1.0 confirms speculation is working)")

In [ ]:

print("Container 'combotest' is still running on port 8080.")
print("Run the quality eval notebooks against http://localhost:8080 before cleaning up.")